# Merton Model for Credit Risk, and the LTCM Case

**UCLA MFE 409 — Financial Risk Management, Problem Set 8** (individual submission)

## Part 1: LTCM — what went wrong

Long-Term Capital Management ran convergence-arbitrage trades: going long an
undervalued security and short a closely related, essentially-equivalent one
(off-the-run vs. on-the-run Treasuries, swap spreads, mortgage-backed vs.
government debt), betting the prices would converge. Because each individual
spread offered only a few basis points of edge, the fund relied on heavy
leverage to turn those spreads into attractive returns — and by 1997, as the
post-Euro convergence opportunities that had driven its early returns dried
up, it returned $2.7B of capital to investors specifically to push leverage
higher for 1998.

Russia's August 1998 debt restructuring (a de facto default) triggered a
global repricing of credit and sovereign risk. Spreads that historically
moved a couple of basis points a day moved by 21bp; LTCM lost $550M on
August 21st alone, and another $550M a month later as equity volatility
spiked. The unwind was compounded by a **margin spiral**: LTCM's prime
broker faced margin calls on LTCM's own futures positions, which drained the
fund's liquidity right as counterparties — uncertain how a Cayman Islands
entity outside U.S. bankruptcy law would behave in default — grew reluctant
to extend further credit.

**The risk-management failure, specifically:** LTCM's VaR framework assumed
(1) constant volatility, when volatility roughly doubled in the crisis, and
(2) a symmetric P&L distribution, which credit-sensitive spread trades don't
have — they look more like a short options position, with limited upside and
a fat left tail. Layer on a fund that had recently shed its most liquid
positions in favor of higher-yielding illiquid ones, and a plausible-looking
daily VaR left the fund badly underprepared for a genuine tail event.

**How I'd manage risk for a similar strategy:** don't rely on point-in-time
VaR alone — factor in *how long it would take to raise capital or unwind a
position* under stress, since that liquidity horizon is exactly what breaks
first. Add expected-shortfall / CVaR alongside VaR to size the tail, not just
its threshold, and use a volatility model (GARCH or similar) that reacts to
regime changes rather than assuming constant variance.

## Part 2: Merton model for credit risk

**Setup.** A company has equity value $E_0 = \$3B$ with equity volatility
$\sigma_E = 50\%$, debt with face value $F = \$20B$ maturing in $T=3$ years,
and a risk-free rate $r = 4.5\%$. Under Merton's model, equity is a call
option on the firm's assets struck at the face value of debt:

$$E_0 = V_0\,N(d_1) - F e^{-rT} N(d_2), \qquad
\sigma_E E_0 = N(d_1)\,\sigma_V V_0$$

$$d_1 = \frac{\ln(V_0/F) + (r + \tfrac12\sigma_V^2)T}{\sigma_V\sqrt{T}}, \qquad
d_2 = d_1 - \sigma_V\sqrt{T}$$

This is a **nonlinear system in the two unknowns** — asset value $V_0$ and
asset volatility $\sigma_V$ — because $d_1, d_2$ depend on both. There's no
closed form, so it's solved numerically.

In [ ]:
import numpy as np
from scipy.optimize import fsolve
from scipy.stats import norm

E0 = 3.0          # equity value in billions
sigma_E = 0.50     # equity volatility
F = 20.0           # face value of debt in billions
T = 3.0            # years to maturity
r = 0.045          # risk-free rate

def equations(vars):
    V0, sigma_V = vars

    d1 = (np.log(V0 / F) + (r + 0.5 * sigma_V**2) * T) / (sigma_V * np.sqrt(T))
    d2 = d1 - sigma_V * np.sqrt(T)

    eq1 = V0 * norm.cdf(d1) - F * np.exp(-r * T) * norm.cdf(d2) - E0
    eq2 = norm.cdf(d1) * sigma_V * V0 - sigma_E * E0

    return [eq1, eq2]

initial_guess = [22.0, 0.15]
V0_sol, sigma_V_sol = fsolve(equations, initial_guess)

d1 = (np.log(V0_sol / F) + (r + 0.5 * sigma_V_sol**2) * T) / (sigma_V_sol * np.sqrt(T))
d2 = d1 - sigma_V_sol * np.sqrt(T)
pd = norm.cdf(-d2)

print(f"Asset value V0 = {V0_sol:.6f} billion")
print(f"Asset volatility sigma_V = {sigma_V_sol:.6f}")
print(f"Distance to default (d2) = {d2:.6f}")
print(f"Probability of default = {pd:.6%}")

**Result:** solving the system gives implied asset value
$V_0 \approx \$20.22B$ and asset volatility $\sigma_V \approx 8.71\%$ —
notice how much lower this is than the 50% equity volatility, since equity
is a *levered* claim on the firm's assets. The distance to default,
$d_2 \approx 0.894$ standard deviations, translates to a **default
probability of about 18.6%** over the 3-year horizon.

### Expected recovery rate

Conditional on default, the expected recovery on the debt follows from the
risk-neutral expectation of the asset value given $V_T < F$:

$$R = \frac{V_0}{F}\, e^{rT}\, \frac{N(-d_1)}{N(-d_2)}$$

In [ ]:
R = (V0_sol / F) * np.exp(r * T) * norm.cdf(-d1) / norm.cdf(-d2)
print("Expected recovery rate =", R)

**Result:** expected recovery rate ≈ **92.3%** of face value conditional
on default. That's a high recovery for a defaulted credit — intuitive here,
since the firm's default probability is driven mostly by moderate asset
volatility on a still-large asset base ($20.2B assets against $20B of debt),
rather than a scenario where assets have collapsed far below the debt claim.
It's a useful reminder that Merton-model default probability and recovery
aren't independent: the same asset-value/volatility pair that produces a
default also pins down how deep in the money the "put" that bondholders are
effectively short really is.